# 🧠 CerebroSpheroid V4: Cloud TPU Full 18-Hour Training Pipeline
### 128 GB HBM • 8 TPU Cores • PyTorch/XLA PJRT • Multi-Task Deep Convergence (bfloat16)

### **Architecture & Capabilities in V4-Small:**
1. **Modern Llama 32,000 Vocabulary**: Byte-fallback BPE with atomic single-digit arithmetic (`0`-`9`) and zero OOV tokens (127M total parameters).
2. **Causal Temporal Sensory Ingestion**: Exponential recency decay + phase velocity momentum vectors replacing order-erasing centroid pooling.
3. **SpheroidOptimizer**: Polar Newton-Schulz Riemannian optimizer with persistent power-iteration spectral radius guard (<1.5 ms) and decoupled row-wise RMS token momentum (<0.5 ms).
4. **Reasoning-Focused Mix**: Multi-step instruction and math reasoning (`OpenHermes=0.45, LMSYS=0.35, GSM8K=0.15, Cognitive=0.05`).
5. **18-Hour Budget & Kaggle 9-Hour Limit**: Kaggle batch runs have a strict 9-hour execution timeout. To train for 18 hours safely, run **Session 1 for 8.5 hours** (`--max_hours 8.5`), which cleanly packages checkpoints. Then run **Session 2 for 8.5 hours** resuming from Session 1 output for a full 17–18 hours without risking timeout termination!

---
### ⚡ Kaggle Settings:
- **Accelerator**: Select **TPU VM v3-8** or **TPU v5e**
- **Internet**: Toggle **Internet ON**
- **Persistence**: Variables and Files


## Step 1: TPU Hardware & PyTorch/XLA PJRT Verification

In [ ]:
import os, sys
# Set modern PJRT runtime for Cloud TPU
os.environ['PJRT_DEVICE'] = 'TPU'

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    world_size = xm.xrt_world_size()
    print(f"✅ Successfully connected to Cloud TPU device: {device} (World Size: {world_size})")
except ImportError:
    print("Installing torch_xla for TPU VM...")
    !pip install -q torch~=2.5.0 torch_xla[tpu]~=2.5.0 -f https://storage.googleapis.com/libtpu-releases/index.html
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f"✅ Successfully installed and connected to Cloud TPU: {device}")


## Step 2: Clone / Pull Latest Repository

In [ ]:
import os
%cd /kaggle/working
if not os.path.exists("reasoning_agent"):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git
%cd /kaggle/working/reasoning_agent
!git fetch origin main
!git reset --hard origin/main
!pip install -q pyarrow fastparquet transformers datasets sentencepiece


## Step 3: Checkpoint Ingestion & Resumption (Automatic)

In [ ]:
import os, glob, shutil
os.makedirs("checkpoints", exist_ok=True)
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            full_p = os.path.join(root, f)
            if f.endswith(".zip") and "cerebro_spheroid" in f:
                print(f"Unpacking checkpoint archive: {f}")
                os.system(f'unzip -n -q "{full_p}" -d checkpoints/')
            elif f.endswith(".pt") and "cerebro_spheroid_v4" in f:
                print(f"Found checkpoint file: {f}")
                shutil.copy2(full_p, os.path.join("checkpoints", f))

# Flatten nested checkpoints_tpu or checkpoints_out if present
for nested in ["checkpoints/checkpoints_tpu", "checkpoints/checkpoints_out"]:
    if os.path.exists(nested):
        for item in os.listdir(nested):
            shutil.move(os.path.join(nested, item), os.path.join("checkpoints", item))
        shutil.rmtree(nested, ignore_errors=True)

print("\nCheckpoints available:")
!ls -lh checkpoints/


## Step 4: Acceptance Verification Tests on TPU VM

In [ ]:
!python3 -u training/test_v4_architecture.py
!python3 -u training/test_spheroid_optimizer.py


## Step 5: Full-Resource Cloud TPU Training (Multi-Core Distributed)
- **Multi-Core**: `torchrun --nproc_per_node=4` utilizing all 8 TensorCores
- **Throughput**: 4 chips $\times$ 32 batch $\times$ 4 accum $\times$ 256 window = **131,072 tokens/step**
- **Session Time Budget**: `--max_hours 8.5` (safely under Kaggle's 9.0-hr hard timeout limit)

In [ ]:
!torchrun --nproc_per_node=4 training/train_v4_full.py \
    --arch v4 \
    --preset v4_small \
    --tokenizer llama \
    --mix "openhermes=0.45,lmsys=0.35,gsm8k=0.15,cognitive=0.05" \
    --optimizer spheroid \
    --device xla \
    --dtype bf16 \
    --lr 0.002 \
    --max_steps 50000 \
    --max_hours 8.5 \
    --batch_size 32 \
    --grad_accum 4 \
    --window 256 \
    --eval_window 128 \
    --eval_interval 200 \
    --save_every 100 \
    --save_prefix cerebro_spheroid_v4_small_tpu


### Alternative: Interactive 18-Hour Single-Run Cell
*(If running directly in an active interactive browser tab without the 9-hr batch cutoff)*

In [ ]:
# To train for 18 continuous hours in an interactive editor session:
# !torchrun --nproc_per_node=4 training/train_v4_full.py \
#     --arch v4 \
#     --preset v4_small \
#     --tokenizer llama \
#     --mix "openhermes=0.45,lmsys=0.35,gsm8k=0.15,cognitive=0.05" \
#     --optimizer spheroid \
#     --device xla \
#     --dtype bf16 \
#     --lr 0.002 \
#     --max_steps 50000 \
#     --max_hours 18.0 \
#     --batch_size 32 \
#     --grad_accum 4 \
#     --window 256 \
#     --eval_window 128 \
#     --eval_interval 200 \
#     --save_every 100 \
#     --save_prefix cerebro_spheroid_v4_small_tpu


## Step 6: Archive Checkpoints for Download

In [ ]:
import os, glob
!mkdir -p /kaggle/working/checkpoints_tpu
!cp -r checkpoints/* /kaggle/working/checkpoints_tpu/ 2>/dev/null || true
!zip -r /kaggle/working/cerebro_spheroid_v4_small_tpu_checkpoints.zip /kaggle/working/checkpoints_tpu
!ls -lh /kaggle/working/*.zip
print("\n✅ Checkpoints successfully archived to /kaggle/working/cerebro_spheroid_v4_small_tpu_checkpoints.zip!")
print("Download it directly from the right-hand Kaggle Output panel.")
